# Vega-Lite and Altair style

Vega-Lite facets with `row`, `column` and `facet` (with `columns`) channels, and makes
scales independent with `resolve_scale`:

| Altair                                          | anytrellis                     |
|-------------------------------------------------|--------------------------------|
| `.facet(row="symbol")`                          | `row="symbol"`                 |
| `.facet(column="island")`                       | `col="island"`                 |
| `.facet(facet="year", columns=3)`               | `wrap="year", cols=3`          |
| `.resolve_scale(y="independent")`               | `share_y=False`                |
| `alt.Color("sex")`                              | `color="sex"`                  |

**Using a plot:** click it, then scroll to zoom along x (Shift+scroll: y; over an axis:
that axis), drag to pan, Shift+drag to zoom to a box, `u` to undo and `r` to reset. Panels
that share a scale zoom and pan together. `y` keeps y fitted to the data in view, and `s`
makes drag a brush that selects (Esc clears); a click on a legend entry selects its colour
(Shift+click: more). Hover a mark for its values. Double-click a panel to see it alone
(Page Up and Page Down: the next ones; Backspace: all again), and press `f` (or the
toolbar's button) for full screen.

In [ ]:
import polars as pl

from anytrellis import Trellis
from anytrellis.datasets import load

## Rows with independent y scales

```python
alt.Chart(stocks).mark_line().encode(x="date:T", y="price:Q").facet(row="symbol") \
    .resolve_scale(y="independent")
```

In [ ]:
stocks = load("stocks")

rows = (
    Trellis(stocks, row="symbol", share_y=False, panel_height=80)
    .line(x="date", y="price", color="symbol", tooltip=["symbol"])
)
rows

## A trellis of grouped bars

The US population by age and sex, a panel per census (Vega-Lite's population example,
faceted).

In [ ]:
census = load("population").filter(pl.col("year").is_in([1900, 1930, 1960, 1990, 2000]))

pyramids = (
    Trellis(census, wrap="year", cols=2, panel_height=150, title="US population by age",
            palette=["#e377c2", "#1f77b4"])
    .order("sex", ["female", "male"])
    .bars(x="age", y="people", color="sex", position="dodge")
)
pyramids

## Columns

```python
alt.Chart(gapminder).mark_point().encode(x="fertility", y="life_expect", color="region") \
    .transform_filter(alt.FieldOneOfPredicate("year", [1955, 1980, 2005])).facet(column="year")
```

In [ ]:
three = load("gapminder").filter(pl.col("year").is_in([1955, 1980, 2005]))

columns = (
    Trellis(three, col="year", panel_height=260)
    .points(x="fertility", y="life_expect", color="region", size=7, tooltip=["country"])
)
columns

## A trellis of heatmaps

```python
alt.Chart(weather).mark_rect() \
    .encode(x="date(date):O", y="month(date):O", color="max(temp_max)") \
    .facet(facet="year(date)", columns=2)
```

`rect()` draws a cell per row, centred on x and y (`width=`/`height=` in data units, one by
default); `x2=`/`y2=` give the corners instead.

In [ ]:
daily = load("seattle_weather").with_columns(
    year=pl.col("date").dt.year(), month=pl.col("date").dt.month(), day=pl.col("date").dt.day()
)

heatmap = (
    Trellis(daily, wrap="year", cols=2, panel_height=150, title="Seattle: daily high (°C)")
    .rect(x="day", y="month", color="temp_max", tooltip=["date"])
)
heatmap

## Areas

`mark_area` under `mark_line`, each stock on its own y scale.

In [ ]:
filled = (
    Trellis(stocks, wrap="symbol", cols=3, share_y=False, title="Stock prices")
    .area(x="date", y="price", color="symbol", opacity=0.5)
    .line(x="date", y="price", color="symbol", tooltip=["symbol"])
)
filled